# **YouTube** Transcript Collector
- For an Ironhack Academic Bootcamp AI chatbot project.

### **Installs**

In [1]:
!pip -q install -U youtube-transcript-api yt-dlp pandas tqdm chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.2/182.2 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 134.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 166.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 109.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 30.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 98.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 136.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### **Imports**

In [2]:
import os
import json
import random
import time
import subprocess
import chromadb

import pandas as pd
from tqdm import tqdm

from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api.proxies import WebshareProxyConfig

In [3]:
from urllib.parse import urlparse, parse_qs

def clean_video_id(s: str) -> str:
    s = str(s).strip()

    # If it's a full URL, extract v=
    if s.startswith("http://") or s.startswith("https://"):
        q = parse_qs(urlparse(s).query)
        if "v" in q and q["v"]:
            return q["v"][0]

    # If it's "VIDEOID&something=..."
    return s.split("&", 1)[0]

In [4]:
from google.colab import userdata

ytt_api = YouTubeTranscriptApi(
    proxy_config=WebshareProxyConfig(
        proxy_username=userdata.get("WEBSHARE_USER"), # <-- Secret key pass for webshare rotating ip proxy.
        proxy_password=userdata.get("WEBSHARE_PASS"),
        filter_ip_locations=["de", "us", "gb", "fr", "ca","sk","jp","pl","id","hk","nl","nz"],
    )
)

### **Target playlists**

In [5]:
PLAYLIST_URL = "https://www.youtube.com/playlist?list=PL10NWKboioZT44tZ2AaewLGmlKOPM04ss"
OUTPUT_CSV = "rawentries_p26_workstations_hu.csv"
PLAYLIST_TAG = "workstations_hu"
CHROMA_PATH = "./chroma_corpus"
COLLECTION_NAME = "yt_transcripts_en"
PLAYLIST_TAG = "workstations_en"

### **Setup** & **Model**

In [6]:


api = ytt_api

client = chromadb.PersistentClient(path=CHROMA_PATH)
col = client.get_or_create_collection(
    COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

### **Core Function**

In [7]:
def is_subtitles_disabled(e: Exception) -> bool:
    return "subtitles are disabled" in str(e).lower()

def is_no_transcript(e: Exception) -> bool:
    return "no transcripts were found" in str(e).lower()

def is_ip_block_error(e: Exception) -> bool:
    msg = str(e).lower()
    return (
        "blocking requests from your ip" in msg
        or "requestblocked" in msg
        or "ipblocked" in msg
        or "too many requests" in msg
        or "toomanyrequests" in msg
    )

def fetch_with_backoff(video_id: str, languages=("en",), max_retries=3):
    base_sleep = 35.0
    multiplier = 2.5

    for attempt in range(max_retries):
        try:
            return api.fetch(video_id, languages=list(languages))
        except Exception as e:
            # If it's NOT an IP-block style error, bubble up
            if not is_ip_block_error(e):
                raise

            sleep_s = (base_sleep * (multiplier ** attempt)) + random.uniform(0, 1.5)
            print(f"[BLOCKED] {video_id} — sleeping {sleep_s:.1f}s (attempt {attempt+1}/{max_retries})")
            time.sleep(sleep_s)

    raise RuntimeError(f"Still blocked after {max_retries} retries for {video_id}")

def get_playlist_entries(playlist_url: str):
    """
    Returns a list of dicts:
    [{"id": "...", "title": "...", "url": "..."}]
    Uses yt-dlp.
    """
    cmd = [
        "yt-dlp",
        "--flat-playlist",
        "--dump-single-json",
        playlist_url,
    ]
    out = subprocess.check_output(cmd, text=True)
    data = json.loads(out)

    entries = []
    for e in data.get("entries", []):
        vid = e.get("id")
        if not vid:
            continue
        entries.append({
            "id": vid,
            "title": e.get("title", ""),
            "url": f"https://www.youtube.com/watch?v={vid}",
        })
    return entries

def merge_transcript_chunks(chunks, max_chars=900, min_chars=300):
    """
    Merge micro transcript segments into RAG-sized
    chunks while preserving timestamps.
    """
    merged = []
    buf_text = []
    buf_start = None
    buf_end = None

    for chunk in chunks:
        t = chunk["text"].strip()
        if not t:
            continue
        if buf_start is None:
            buf_start = chunk["start"]

        candidate = (" ".join(buf_text + [t])).strip()

        if len(candidate) < max_chars:
            buf_text.append(t)
            buf_end = chunk["end"]
        else:
            if buf_text:
                merged.append({
                    "text": " ".join(buf_text).strip(),
                    "start": buf_start,
                    "end": buf_end
                })
            buf_text = [t]
            buf_start = chunk["start"]
            buf_end = chunk["end"]

    if buf_text:
        merged.append({
            "text": " ".join(buf_text).strip(),
            "start": buf_start,
            "end": buf_end
        })

    merged = [m for m in merged if len(m["text"]) >= min_chars]
    return merged

### **Load Entries**

In [8]:
entries = get_playlist_entries(PLAYLIST_URL)
print("Playlist videos found:", len(entries))
entries[:3]



Playlist videos found: 50


[{'id': 'BhMQa9rEZFo',
  'title': "Threadripper 7995WX 96 Core: How's This System So Small!? Falcon Northwest RAK",
  'url': 'https://www.youtube.com/watch?v=BhMQa9rEZFo'},
 {'id': 'ofwV2kdDdN4',
  'title': 'The New 14700k Fragbox = Amazing Gaming Fun!!',
  'url': 'https://www.youtube.com/watch?v=ofwV2kdDdN4'},
 {'id': 'kjmmdv0HU4o',
  'title': 'Hitting The (Sapphire) Rapids With The ASRock W790',
  'url': 'https://www.youtube.com/watch?v=kjmmdv0HU4o'}]

### **Check indexes**

In [9]:
existing = col.get(include=["metadatas"])

existing_video_ids = set()
for meta in existing["metadatas"]:
    if meta and "video_id" in meta:
        existing_video_ids.add(meta["video_id"])

print("Already indexed videos:", len(existing_video_ids))

Already indexed videos: 0


### **Configure** the fetching process

In [10]:
import os
import json

# --- CONFIG (minimal) ---
PLAYLIST_COUNTER = 26  # <- increment this each time I run a different playlist
RAW_PREFIX = f"rawentries_p{PLAYLIST_COUNTER}_part"
STATE_PATH = f"resume_state_p{PLAYLIST_COUNTER}.json"

SAVE_EVERY_N_VIDEOS = 10
STOP_AFTER_CONSECUTIVE_BLOCKS = 3

# --- RESUME (remember where I stopped) ---
start_idx = 0
if os.path.exists(STATE_PATH):
    with open(STATE_PATH, "r", encoding="utf-8") as f:
        state = json.load(f)
    start_idx = int(state.get("next_index", 0))

print(f"Resume enabled — starting at entry index: {start_idx} / {len(entries)}")
print(f"Output files will be named: {RAW_PREFIX}1.csv, {RAW_PREFIX}2.csv, ...")

# --- Batch buffers ---
raw_rows = []          # keep using your raw_rows structure
failed_rows = []       # keep using your failed_rows structure
batch_counter = 1
videos_processed_in_batch = 0
consecutive_blocked = 0

Resume enabled — starting at entry index: 0 / 50
Output files will be named: rawentries_p26_part1.csv, rawentries_p26_part2.csv, ...


### **Fetch Transcipts**

In [11]:
from tqdm import tqdm

def _save_batch_csv(raw_rows, failed_rows, batch_counter):
    raw_path = f"{RAW_PREFIX}{batch_counter}.csv"
    fail_path = f"{RAW_PREFIX}{batch_counter}_failed.csv"

    if raw_rows:
        pd.DataFrame(raw_rows).to_csv(raw_path, index=False)
        print(f"[SAVED] {raw_path} — rows: {len(raw_rows)}")
    else:
        print(f"[SAVED] {raw_path} — rows: 0 (no successful transcript rows)")

    if failed_rows:
        pd.DataFrame(failed_rows).to_csv(fail_path, index=False)
        print(f"[SAVED] {fail_path} — rows: {len(failed_rows)}")

def _save_state(next_index, last_video_id=None, last_title=None, reason=None):
    state = {
        "playlist_counter": PLAYLIST_COUNTER,
        "next_index": int(next_index),
        "last_video_id": last_video_id,
        "last_title": last_title,
        "reason": reason,
    }
    with open(STATE_PATH, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)
    print(f"[STATE] wrote {STATE_PATH}: next_index={next_index}, reason={reason}")

for idx, e in enumerate(tqdm(entries[start_idx:], desc="Fetching transcripts"), start=start_idx):
    video_id = e["id"]
    title = e.get("title", "")
    url = e.get("url", f"https://www.youtube.com/watch?v={video_id}")

    # ---- your existing "already indexed" check can stay above this loop, unchanged ----
    # ---- if I also have existing_video_ids, keep that check here unchanged ----
    # Example (keep if you already have it):
    # if video_id in existing_video_ids:
    #     continue

    try:
        transcript_en = fetch_with_backoff(video_id, languages=("en",), max_retries=3)
        consecutive_blocked = 0  # reset on success
    except Exception as ex:
        # Track failure row (minimal, compatible with your earlier patterns)
        failed_rows.append({
            "video_id": video_id,
            "title": title,
            "url": url,
            "error": str(ex),
            "subtitles_disabled": is_subtitles_disabled(ex),
            "no_transcript": is_no_transcript(ex),
            "ip_block_like": is_ip_block_error(ex),
        })
        print(f"[{idx+1}/{len(entries)}] FAIL {video_id}: {ex}")

        # Stop condition: 3 blocked videos in a row (different videos)
        if is_ip_block_error(ex):
            consecutive_blocked += 1
            if consecutive_blocked >= STOP_AFTER_CONSECUTIVE_BLOCKS:
                # Save whatever we have before stopping
                _save_batch_csv(raw_rows, failed_rows, batch_counter)
                _save_state(
                    next_index=idx + 1,
                    last_video_id=video_id,
                    last_title=title,
                    reason=f"stopped_after_{STOP_AFTER_CONSECUTIVE_BLOCKS}_consecutive_ip_blocks",
                )
                print("[STOP] Too many consecutive IP blocks. Stop now and resume later.")
                break
        else:
            # non-block failure does not count toward consecutive blocked
            consecutive_blocked = 0

        # Count this as a processed video attempt for batching
        videos_processed_in_batch += 1
        if videos_processed_in_batch >= SAVE_EVERY_N_VIDEOS:
            _save_batch_csv(raw_rows, failed_rows, batch_counter)
            _save_state(next_index=idx + 1, last_video_id=video_id, last_title=title, reason="batch_saved")
            batch_counter += 1
            raw_rows = []
            failed_rows = []
            videos_processed_in_batch = 0
        continue

    # ✅ polite pacing between successful requests (your pattern)
    time.sleep(random.uniform(4.0, 15.0))

    # Convert transcript objects/segments into raw rows (keep your exact per-chunk logic style)
    for i, chunk in enumerate(transcript_en):
        # Support both object-style (chunk.text) and dict-style (chunk["text"])
        text = getattr(chunk, "text", None) or (chunk.get("text") if isinstance(chunk, dict) else None) or ""
        text = text.strip()
        if not text:
            continue

        start = float(getattr(chunk, "start", None) or (chunk.get("start") if isinstance(chunk, dict) else 0.0) or 0.0)
        duration = float(getattr(chunk, "duration", None) or (chunk.get("duration") if isinstance(chunk, dict) else 0.0) or 0.0)
        end = start + max(duration, 0.0)

        raw_rows.append({
            "segment_id": f"{video_id}_{i:06d}",
            "video_id": video_id,
            "title": title,
            "url": url,
            "lang": "en",
            "t_start": start,
            "t_end": end,
            "duration": duration,
            "text": text,
            "playlist": PLAYLIST_TAG,
        })

    print(f"[{idx+1}/{len(entries)}] OK {video_id} — raw lines: {len(transcript_en)} — {title}")

    # Count this video as processed for batching
    videos_processed_in_batch += 1
    if videos_processed_in_batch >= SAVE_EVERY_N_VIDEOS:
        _save_batch_csv(raw_rows, failed_rows, batch_counter)
        _save_state(next_index=idx + 1, last_video_id=video_id, last_title=title, reason="batch_saved")
        batch_counter += 1
        raw_rows = []
        failed_rows = []
        videos_processed_in_batch = 0

else:
    # Loop completed normally: save remainder + mark complete
    _save_batch_csv(raw_rows, failed_rows, batch_counter)
    _save_state(next_index=len(entries), reason="completed_playlist")
    print("[DONE] Playlist finished. State saved as completed.")

Fetching transcripts:   2%|▏         | 1/50 [00:16<13:35, 16.63s/it]

[1/50] OK BhMQa9rEZFo — raw lines: 394 — Threadripper 7995WX 96 Core: How's This System So Small!? Falcon Northwest RAK


Fetching transcripts:   4%|▍         | 2/50 [00:38<15:48, 19.76s/it]

[2/50] OK ofwV2kdDdN4 — raw lines: 530 — The New 14700k Fragbox = Amazing Gaming Fun!!


Fetching transcripts:   6%|▌         | 3/50 [00:56<14:44, 18.82s/it]

...

[48/50] OK chL2fBZUh8c — raw lines: 127 — The Epiphan Pearl 2 and the Rise of Level2Techs


Fetching transcripts:  98%|█████████▊| 49/50 [15:36<00:22, 22.81s/it]

[49/50] OK kAk3Uyhg5UI — raw lines: 223 — Epiphan Webcaster X1: Youtube Streaming Appliance -- Stream without a Computer!


Fetching transcripts: 100%|██████████| 50/50 [16:00<00:00, 19.20s/it]

[50/50] OK fZBc5Eck3XE — raw lines: 428 — Level1Reviews: Thecus W2810 Pro
[SAVED] rawentries_p26_part5.csv — rows: 2674
[SAVED] rawentries_p26_part5_failed.csv — rows: 1
[STATE] wrote resume_state_p26.json: next_index=50, reason=batch_saved
[SAVED] rawentries_p26_part6.csv — rows: 0 (no successful transcript rows)
[STATE] wrote resume_state_p26.json: next_index=50, reason=completed_playlist
[DONE] Playlist finished. State saved as completed.


### **Merge Chunks** - Offline

In [12]:
import os
import glob
import pandas as pd

# ============================================================
# BATCH MERGE — processes all raw part files at once
# ============================================================
RAW_PATTERN = "rawentries_p26_part*.csv"    # ← adjust the p-number per playlist
OUTPUT_CSV = "rawentries_P26_workstations_chunked.csv"  # ← final chunked output
MAX_CHARS = 600
MIN_CHARS = 300

# Find all matching part files (exclude _failed and _chunked files)
all_files = sorted(glob.glob(RAW_PATTERN))
all_files = [f for f in all_files if "_failed" not in f and "_chunked" not in f]

print(f"Found {len(all_files)} part files:")
for f in all_files:
    print(f"  {f}")

# Load and combine all parts
dfs = []
for f in all_files:
    df = pd.read_csv(f)
    dfs.append(df)
    print(f"  ✅ {f} — {len(df)} rows")

raw_df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal raw segments: {len(raw_df)}")
print(f"Unique videos: {raw_df['video_id'].nunique()}")

# Validate required columns
required_cols = {"video_id", "title", "t_start", "t_end", "text"}
missing = required_cols - set(raw_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

raw_df["t_start"] = raw_df["t_start"].astype(float)
raw_df["t_end"] = raw_df["t_end"].astype(float)
raw_df["text"] = raw_df["text"].fillna("").astype(str)
raw_df = raw_df.sort_values(["video_id", "t_start"])

# Chunk each video
chunked_rows = []

for video_id, g in raw_df.groupby("video_id", sort=False):
    title = str(g["title"].iloc[0]) if "title" in g.columns else ""
    playlist = str(g["playlist"].iloc[0]) if "playlist" in g.columns else ""
    lang = str(g["lang"].iloc[0]) if "lang" in g.columns else "en"
    url = str(g["url"].iloc[0]) if "url" in g.columns else f"https://www.youtube.com/watch?v={video_id}"

    structured_chunks = [
        {"text": row["text"], "start": float(row["t_start"]), "end": float(row["t_end"])}
        for _, row in g.iterrows()
    ]

    merged = merge_transcript_chunks(chunks=structured_chunks, max_chars=MAX_CHARS, min_chars=MIN_CHARS)

    for i, c in enumerate(merged):
        chunked_rows.append({
            "chunk_id": f"{video_id}_{i}",
            "video_id": video_id,
            "title": title,
            "lang": lang,
            "t_start": c["start"],
            "t_end": c["end"],
            "text": c["text"],
            "url": url,
            "playlist": playlist,
        })

chunked_df = pd.DataFrame(chunked_rows)
chunked_df.to_csv(OUTPUT_CSV, index=False)

print(f"\n{'='*50}")
print(f"✅ Saved: {OUTPUT_CSV}")
print(f"   Chunks: {len(chunked_df)}")
print(f"   Videos: {chunked_df['video_id'].nunique()}")
print(f"{'='*50}")

Found 5 part files:
  rawentries_p26_part1.csv
  rawentries_p26_part2.csv
  rawentries_p26_part3.csv
  rawentries_p26_part4.csv
  rawentries_p26_part5.csv
  ✅ rawentries_p26_part1.csv — 4974 rows
  ✅ rawentries_p26_part2.csv — 4073 rows
  ✅ rawentries_p26_part3.csv — 4674 rows
  ✅ rawentries_p26_part4.csv — 3322 rows
  ✅ rawentries_p26_part5.csv — 2674 rows

Total raw segments: 19717
Unique videos: 48

✅ Saved: rawentries_P26_workstations_chunked.csv
   Chunks: 1194
   Videos: 48


### Check output file

In [ ]:
print(f"Videos:             {chunked_df['video_id'].nunique()}")
print(f"Total chunks:       {len(chunked_df)}")
print(f"Avg chars/chunk:    {int(chunked_df['text'].str.len().mean())}")
print(f"Min chars:          {int(chunked_df['text'].str.len().min())}")
print(f"Max chars:          {int(chunked_df['text'].str.len().max())}")

bad = (chunked_df["t_end"] <= chunked_df["t_start"]).sum()
print(f"Bad timestamps:     {bad}")

# Show coverage by playlist tag
print(f"\nPlaylist tags:")
print(chunked_df["playlist"].value_counts().to_string())

Videos:             101
Total chunks:       1915
Avg chars/chunk:    575
Min chars:          311
Max chars:          599
Bad timestamps:     0

Playlist tags:
playlist
pc_builds_en    1915


### **Merge Chunks** - online

In [ ]:
for idx, e in enumerate(tqdm(entries, desc="Indexing merged chunks"), start=1):
    video_id = e["id"]
    title = e["title"]

    if video_id in existing_video_ids:
        continue

    try:
        transcript_en = fetch_with_backoff(video_id, languages=("en",))
    except Exception as ex:
        print(f"SKIP {video_id}: {ex}")
        continue

    time.sleep(random.uniform(1.5, 10.0))

    structured_chunks = [
        {"text": chunk.text,
         "start": chunk.start,
         "end": chunk.start + chunk.duration}
        for chunk in transcript_en
    ]

    merged_chunks = merge_transcript_chunks(chunks=structured_chunks, max_chars=600)
    if not merged_chunks:
        print(f"[{idx}/{len(entries)}] SKIP {video_id} — merged_chunks empty")
        continue

    docs = [c["text"] for c in merged_chunks]
    metas = [
        {
            "video_id": video_id,
            "title": title,
            "lang": "en",
            "start": c["start"],
            "end": c["end"],
            "playlist": PLAYLIST_TAG,
        }
        for c in merged_chunks
    ]

    embs = model.encode(
        docs,
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True
    ).tolist()

    ids = [f"{video_id}_{i}" for i in range(len(docs))]

    if hasattr(col, "upsert"):
        col.upsert(
            documents=docs,
            embeddings=embs,
            ids=ids,
            metadatas=metas
        )
    else:
        col.add(
            documents=docs,
            embeddings=embs,
            ids=ids,
            metadatas=metas
        )

    print(f"[{idx}/{len(entries)}] OK {video_id} — {len(docs)} chunks — {title}")

### Convert tsv > csv (external data handler)

In [ ]:
import csv

tsv_path = "rawentries_pl.csv"      # change if needed
csv_path = "rawentries_pl.csv"

with open(tsv_path, "r", encoding="utf-8", newline="") as fin, \
     open(csv_path, "w", encoding="utf-8", newline="") as fout:
    reader = csv.reader(fin, delimiter="\t")
    writer = csv.writer(fout)
    writer.writerows(reader)

print("Saved:", csv_path)